In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
spark = SparkSession.builder \
    .appName("Ecommerce Case Study") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 3.5.6


In [9]:
import os

for root, dirs, files in os.walk("data"):
    print(root)
    for f in files:
        print("   ", f)

In [10]:
customers_df = spark.read.csv("customers/*.csv", header=True, inferSchema=True)

products_df = spark.read.csv("products/*.csv", header=True, inferSchema=True)

orders_df = spark.read.csv("orders/*.csv", header=True, inferSchema=True)

order_items_df = spark.read.csv("order_items/*.csv", header=True, inferSchema=True)

returns_df = spark.read.csv("returns/*.csv", header=True, inferSchema=True)

customers_df.show(5)

products_df.show(5)

orders_df.show(5)

print("Customers:", customers_df.count())
print("Products :", products_df.count())
print("Orders   :", orders_df.count())
print("Returns  :", returns_df.count())

+-----------+-------------+--------+-----+-----------------+----------------+
|customer_id|customer_name|    city|state|registration_date|customer_segment|
+-----------+-------------+--------+-----+-----------------+----------------+
|          1|   Customer_1|Columbus|   OH|       2023-10-17|             VIP|
|          2|   Customer_2|   Miami|   CA|       2022-04-25|         Premium|
|          3|   Customer_3| Atlanta|   FL|       2022-01-26|         Premium|
|          4|   Customer_4| Chicago|   OH|       2022-10-09|        Standard|
|          5|   Customer_5|Columbus|   IL|       2022-09-08|         Premium|
+-----------+-------------+--------+-----+-----------------+----------------+
only showing top 5 rows

+----------+------------+--------------+-------+---------+
|product_id|product_name|      category|  brand|unit_cost|
+----------+------------+--------------+-------+---------+
|         1|   Product_1|Home & Kitchen|Brand_A|   509.39|
|         2|   Product_2|   Electroni

In [11]:
sales_by_category = (
    order_items_df
    .join(products_df, "product_id")
    .withColumn(
        "sales_amount",
        col("quantity") * col("selling_price")
    )
    .groupBy("category")
    .agg(
        round(sum("sales_amount"), 2).alias("total_sales")
    )
    .orderBy(desc("total_sales"))
)

sales_by_category.show(truncate=False)

[Stage 61:====>                                                   (1 + 11) / 12]

+--------------+--------------+
|category      |total_sales   |
+--------------+--------------+
|Beauty        |7.626693059E8 |
|Home & Kitchen|7.5813887328E8|
|Books         |7.4649077835E8|
|Toys          |7.446190723E8 |
|Electronics   |7.4426650411E8|
|Sports        |7.4333886813E8|
|Clothing      |7.4192279457E8|
+--------------+--------------+



In [12]:
top_customers = (
    orders_df
    .join(order_items_df, "order_id")
    .join(customers_df, "customer_id")
    .withColumn(
        "purchase_amount",
        col("quantity") * col("selling_price")
    )
    .groupBy(
        "customer_id",
        "customer_name"
    )
    .agg(
        round(sum("purchase_amount"), 2).alias("total_purchase")
    )
    .orderBy(desc("total_purchase"))
    .limit(10)
)

top_customers.show(truncate=False)

[Stage 69:===========================================>            (10 + 3) / 13]

+-----------+--------------+--------------+
|customer_id|customer_name |total_purchase|
+-----------+--------------+--------------+
|93094      |Customer_93094|181569.68     |
|64560      |Customer_64560|169060.4      |
|23289      |Customer_23289|161573.8      |
|52275      |Customer_52275|153364.79     |
|61218      |Customer_61218|153067.55     |
|52034      |Customer_52034|152680.05     |
|40442      |Customer_40442|151037.32     |
|60528      |Customer_60528|148691.95     |
|84830      |Customer_84830|148363.84     |
|82593      |Customer_82593|148281.04     |
+-----------+--------------+--------------+



In [13]:
latest_year = (
    orders_df
    .select(max(year("order_date")).alias("max_year"))
    .collect()[0]["max_year"]
)

monthly_sales = (
    orders_df
    .join(order_items_df, "order_id")
    .filter(year("order_date") == latest_year)
    .withColumn(
        "sales_amount",
        col("quantity") * col("selling_price")
    )
    .groupBy(month("order_date").alias("month"))
    .agg(
        round(sum("sales_amount"), 2).alias("monthly_sales")
    )
    .orderBy("month")
)

monthly_sales.show()

[Stage 81:============>                                           (3 + 10) / 13]

+-----+--------------+
|month| monthly_sales|
+-----+--------------+
|    1|4.4457777576E8|
|    2| 4.153661442E8|
|    3|4.4362824541E8|
|    4|4.2782097434E8|
|    5|4.4481061895E8|
|    6|4.3170515406E8|
|    7|4.4367051912E8|
|    8|4.4109517702E8|
|    9|4.3107152608E8|
|   10|4.4136378931E8|
|   11|4.3362336404E8|
|   12|4.4271290835E8|
+-----+--------------+



In [14]:
total_orders_category = (
    order_items_df
    .join(products_df, "product_id")
    .groupBy("category")
    .agg(count("*").alias("total_orders"))
)

returned_orders_category = (
    returns_df
    .join(order_items_df, "order_id")
    .join(products_df, "product_id")
    .groupBy("category")
    .agg(count("*").alias("returned_orders"))
)

return_percentage = (
    total_orders_category
    .join(
        returned_orders_category,
        "category",
        "left"
    )
    .fillna(0)
    .withColumn(
        "return_percentage",
        round(
            (col("returned_orders") /
             col("total_orders")) * 100,
            2
        )
    )
)

return_percentage.show(truncate=False)

+--------------+------------+---------------+-----------------+
|category      |total_orders|returned_orders|return_percentage|
+--------------+------------+---------------+-----------------+
|Home & Kitchen|434034      |43418          |10.0             |
|Sports        |424412      |42530          |10.02            |
|Electronics   |425896      |42601          |10.0             |
|Clothing      |427607      |42660          |9.98             |
|Books         |427086      |42809          |10.02            |
|Beauty        |430547      |43194          |10.03            |
|Toys          |430418      |43382          |10.08            |
+--------------+------------+---------------+-----------------+



In [15]:
payment_stats = (
    orders_df
    .join(customers_df, "customer_id")
    .groupBy(
        "state",
        "payment_mode"
    )
    .agg(
        count("*").alias("total_orders")
    )
)

window_spec = Window.partitionBy("state") \
                    .orderBy(desc("total_orders"))

preferred_payment = (
    payment_stats
    .withColumn(
        "rank",
        row_number().over(window_spec)
    )
    .filter(col("rank") == 1)
    .select(
        "state",
        "payment_mode",
        "total_orders"
    )
)

preferred_payment.show(truncate=False)

+-----+----------------+------------+
|state|payment_mode    |total_orders|
+-----+----------------+------------+
|CA   |UPI             |20246       |
|FL   |Debit Card      |20010       |
|GA   |Net Banking     |20041       |
|IL   |Cash on Delivery|20498       |
|MI   |Debit Card      |20416       |
|NC   |Net Banking     |19596       |
|NY   |Debit Card      |20369       |
|OH   |Net Banking     |20351       |
|TX   |UPI             |20065       |
|WA   |UPI             |20244       |
+-----+----------------+------------+



In [16]:
customer_analysis = (
    orders_df
    .join(order_items_df, "order_id")
    .join(products_df, "product_id")
    .join(customers_df, "customer_id")
    .withColumn(
        "amount",
        col("quantity") * col("selling_price")
    )
    .groupBy(
        "customer_id",
        "customer_name"
    )
    .agg(
        countDistinct("category").alias("categories_bought"),
        round(sum("amount"), 2).alias("total_spent")
    )
    .filter(
        (col("categories_bought") >= 5) &
        (col("total_spent") > 100000)
    )
)

customer_analysis.show(truncate=False)

26/06/15 17:06:53 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/15 17:06:53 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/15 17:06:53 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/15 17:06:53 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/15 17:06:53 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/15 17:06:53 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/15 17:06:53 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/15 17:06:53 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/15 17:06:54 WARN RowBasedKeyValueBatch: Calling spill() on

+-----------+--------------+-----------------+-----------+
|customer_id|customer_name |categories_bought|total_spent|
+-----------+--------------+-----------------+-----------+
|52297      |Customer_52297|7                |107812.68  |
|26241      |Customer_26241|7                |121047.8   |
|41157      |Customer_41157|7                |105187.1   |
|18149      |Customer_18149|7                |101780.7   |
|46060      |Customer_46060|7                |115805.04  |
|90203      |Customer_90203|7                |109729.63  |
|97920      |Customer_97920|7                |117591.17  |
|28078      |Customer_28078|7                |115138.03  |
|67631      |Customer_67631|7                |106734.26  |
|68399      |Customer_68399|7                |119426.7   |
|60620      |Customer_60620|7                |110953.51  |
|76094      |Customer_76094|7                |119256.5   |
|17867      |Customer_17867|7                |105736.33  |
|27963      |Customer_27963|7                |115995.79 

In [17]:
product_revenue = (
    order_items_df
    .join(products_df, "product_id")
    .withColumn(
        "revenue",
        col("quantity") * col("selling_price")
    )
    .groupBy(
        "category",
        "product_id",
        "product_name"
    )
    .agg(
        round(sum("revenue"), 2).alias("total_revenue")
    )
)

window_spec = Window.partitionBy("category") \
                    .orderBy(desc("total_revenue"))

top_products = (
    product_revenue
    .withColumn(
        "rank",
        row_number().over(window_spec)
    )
    .filter(col("rank") <= 3)
    .orderBy("category", "rank")
)

top_products.show(truncate=False)

[Stage 121:==================>                                     (4 + 8) / 12]

+--------------+----------+-------------+-------------+----+
|category      |product_id|product_name |total_revenue|rank|
+--------------+----------+-------------+-------------+----+
|Beauty        |44016     |Product_44016|277567.99    |1   |
|Beauty        |14849     |Product_14849|274894.2     |2   |
|Beauty        |786       |Product_786  |272174.7     |3   |
|Books         |35314     |Product_35314|296468.78    |1   |
|Books         |28311     |Product_28311|286757.72    |2   |
|Books         |37479     |Product_37479|276736.71    |3   |
|Clothing      |7025      |Product_7025 |293821.97    |1   |
|Clothing      |1560      |Product_1560 |288474.09    |2   |
|Clothing      |31322     |Product_31322|282241.17    |3   |
|Electronics   |6719      |Product_6719 |299113.87    |1   |
|Electronics   |23519     |Product_23519|289561.72    |2   |
|Electronics   |38170     |Product_38170|288875.23    |3   |
|Home & Kitchen|5012      |Product_5012 |305836.22    |1   |
|Home & Kitchen|37452   